In [2]:
!pip -q install openai datasets pandas tqdm tenacity

In [3]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
print("API key loaded securely.")

Enter your OpenAI API key: ··········
API key loaded securely.


In [6]:
import os
import re
import json
import time
import pandas as pd
from tqdm import tqdm
from collections import Counter
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

from datasets import load_dataset
from openai import OpenAI, RateLimitError

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "gpt-5.5"
DATASET_NAME = "ChanceFocus/flare-finred"

# First test with 5.
# Then set LIMIT = None for the full test set.
LIMIT = 100

dataset = load_dataset(DATASET_NAME)

split_name = "test" if "test" in dataset else list(dataset.keys())[0]
split_data = dataset[split_name]

print(dataset)
print("Split:", split_name)
print("Columns:", split_data.column_names)
print(split_data[0])


def parse_triplet_string(s):
    parts = [p.strip() for p in str(s).split(";")]
    if len(parts) != 3:
        return None

    head, tail, relation = parts

    if not head or not tail or not relation:
        return None

    return {
        "head": head,
        "tail": tail,
        "relation": relation
    }


def parse_gold_triples(label_value):
    triples = []

    if isinstance(label_value, list):
        items = label_value
    else:
        items = [label_value]

    for item in items:
        parsed = parse_triplet_string(item)
        if parsed is not None:
            triples.append(parsed)

    return triples


def extract_relations_from_query(query):
    relations = re.findall(r"'([^']+)'", str(query))
    return sorted(set([r.strip() for r in relations if r.strip()]))


# Build relation labels from query and gold labels.
query_relations = extract_relations_from_query(split_data[0]["query"])

gold_relations = set()
for row in split_data:
    for triple in parse_gold_triples(row["label"]):
        gold_relations.add(triple["relation"])

RELATION_LABELS = sorted(set(query_relations) | gold_relations)

print("Number of relation labels:", len(RELATION_LABELS))
print("Relation labels:", RELATION_LABELS)


TRIPLET_SCHEMA = {
    "type": "object",
    "properties": {
        "triplets": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "head": {
                        "type": "string"
                    },
                    "tail": {
                        "type": "string"
                    },
                    "relation": {
                        "type": "string",
                        "enum": RELATION_LABELS
                    }
                },
                "required": ["head", "tail", "relation"],
                "additionalProperties": False
            }
        }
    },
    "required": ["triplets"],
    "additionalProperties": False
}


SYSTEM_PROMPT = """
You are a financial relation extraction model.

Task:
Given a financial sentence, extract all relation triplets.

Each triplet must contain:
- head: the exact head entity text from the sentence
- tail: the exact tail entity text from the sentence
- relation: one label from the allowed relation list

Rules:
1. Extract all valid triplets from the sentence.
2. Use exact entity text from the sentence.
3. Do not invent entities.
4. Do not invent relation labels.
5. If there is no valid triplet, return an empty list.
6. Return only valid JSON.
"""


@retry(
    retry=retry_if_exception_type(RateLimitError),
    wait=wait_exponential(multiplier=5, min=10, max=120),
    stop=stop_after_attempt(10)
)
def call_openai_finred(text):
    allowed_relations_text = ", ".join([f"'{r}'" for r in RELATION_LABELS])

    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    f"Allowed relation labels:\n{allowed_relations_text}\n\n"
                    f"Sentence:\n{text}\n\n"
                    "Return all relation triplets."
                )
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "finred_relation_extraction_result",
                "schema": TRIPLET_SCHEMA,
                "strict": True
            }
        },
        reasoning={
            "effort": "low"
        },
        max_output_tokens=1000
    )

    return json.loads(response.output_text)


def normalize_entity(s):
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = s.strip(" \"'.,;:")
    return s


def normalize_relation(s):
    s = str(s).strip().lower()
    s = re.sub(r"\s+", "_", s)
    return s


def normalize_triplet(triple):
    return (
        normalize_entity(triple.get("head", "")),
        normalize_entity(triple.get("tail", "")),
        normalize_relation(triple.get("relation", ""))
    )


def clean_predicted_triples(predicted_triples):
    cleaned = []

    for item in predicted_triples:
        if not isinstance(item, dict):
            continue

        head = str(item.get("head", "")).strip()
        tail = str(item.get("tail", "")).strip()
        relation = str(item.get("relation", "")).strip()

        if head and tail and relation:
            cleaned.append({
                "head": head,
                "tail": tail,
                "relation": relation
            })

    return cleaned


def compute_metrics(gold_all, pred_all):
    total_tp = 0
    total_fp = 0
    total_fn = 0
    exact_match_count = 0

    for gold_triples, pred_triples in zip(gold_all, pred_all):
        gold_counter = Counter([normalize_triplet(t) for t in gold_triples])
        pred_counter = Counter([normalize_triplet(t) for t in pred_triples])

        tp = sum((gold_counter & pred_counter).values())
        fp = sum((pred_counter - gold_counter).values())
        fn = sum((gold_counter - pred_counter).values())

        total_tp += tp
        total_fp += fp
        total_fn += fn

        if gold_counter == pred_counter:
            exact_match_count += 1

    precision = total_tp / (total_tp + total_fp) if total_tp + total_fp > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if total_tp + total_fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    exact_match_accuracy = exact_match_count / len(gold_all) if len(gold_all) > 0 else 0.0

    triplet_accuracy = (
        total_tp / (total_tp + total_fp + total_fn)
        if total_tp + total_fp + total_fn > 0
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "exact_match_accuracy": exact_match_accuracy,
        "triplet_accuracy": triplet_accuracy,
        "true_positive": total_tp,
        "false_positive": total_fp,
        "false_negative": total_fn
    }


eval_data = split_data if LIMIT is None else split_data.select(range(LIMIT))

rows = []
gold_all = []
pred_all = []

for idx, row in enumerate(tqdm(eval_data)):
    text = row["text"]
    gold_triples = parse_gold_triples(row["label"])

    try:
        result = call_openai_finred(text)
        pred_triples = clean_predicted_triples(result.get("triplets", []))
        error = ""
    except Exception as e:
        pred_triples = []
        error = str(e)
        print(f"Error at row {idx}: {e}")

    gold_all.append(gold_triples)
    pred_all.append(pred_triples)

    gold_counter = Counter([normalize_triplet(t) for t in gold_triples])
    pred_counter = Counter([normalize_triplet(t) for t in pred_triples])

    rows.append({
        "idx": idx,
        "id": row.get("id", ""),
        "text": text,
        "gold_answer": row.get("answer", ""),
        "gold_triples": gold_triples,
        "predicted_triples": pred_triples,
        "exact_match": gold_counter == pred_counter,
        "error": error
    })

    # Slow down to avoid RateLimitError.
    time.sleep(5)


metrics = compute_metrics(gold_all, pred_all)

print("\n===== GPT-5.5 FLARE-FinRED Evaluation =====")
print(f"Dataset: {DATASET_NAME}")
print(f"Split: {split_name}")
print(f"Model: {MODEL}")
print(f"Samples evaluated: {len(eval_data)}")
print(f"Precision / Correctness Rate: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"Exact Match Accuracy: {metrics['exact_match_accuracy']:.4f}")
print(f"Triplet Accuracy: {metrics['triplet_accuracy']:.4f}")
print(f"TP: {metrics['true_positive']}")
print(f"FP: {metrics['false_positive']}")
print(f"FN: {metrics['false_negative']}")

df = pd.DataFrame(rows)
df.to_csv("gpt55_flare_finred_evaluation_results.csv", index=False)

with open("gpt55_flare_finred_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

num_errors = df["error"].astype(str).str.len().gt(0).sum()
print(f"Failed samples: {num_errors}")
print(f"Failure rate: {num_errors / len(df):.4f}")

pd.set_option("display.max_colwidth", 120)
df.head()

DatasetDict({
    test: Dataset({
        features: ['id', 'query', 'text', 'answer', 'label'],
        num_rows: 1068
    })
})
Split: test
Columns: ['id', 'query', 'text', 'answer', 'label']
{'id': 'finred0', 'query': "Given the following sentence, identify the head, tail, and relation of each triplet present in the sentence. The relations you should be looking for are 'legal_form', 'publisher', 'owner_of', 'employer', 'manufacturer', 'position_held', 'chairperson', 'industry', 'business_division', 'creator', 'original_broadcaster', 'chief_executive_officer', 'location_of_formation', 'operator', 'owned_by', 'founded_by', 'parent_organization', 'member_of', 'product_or_material_produced', 'brand', 'headquarters_location', 'director_/_manager', 'distribution_format', 'distributed_by', 'platform', 'currency', 'subsidiary', 'stock_exchange', and 'developer'. If a relation exists between two entities, provide your answer in the format 'head ; tail ; rel'. If there are multiple triplets in

100%|██████████| 100/100 [11:51<00:00,  7.12s/it]


===== GPT-5.5 FLARE-FinRED Evaluation =====
Dataset: ChanceFocus/flare-finred
Split: test
Model: gpt-5.5
Samples evaluated: 100
Precision / Correctness Rate: 0.0828
Recall: 0.1085
F1 Score: 0.0940
Exact Match Accuracy: 0.0500
Triplet Accuracy: 0.0493
TP: 14
FP: 155
FN: 115
Failed samples: 0
Failure rate: 0.0000


,idx,id,text,gold_answer,gold_triples,predicted_triples,exact_match,error
0,0,finred0,"Wednesday, July 8, 2015 10:30AM IST (5:00AM GMT) Rimini Street Comment on Oracle Litigation Las Vegas, United States...",PeopleSoft ; JD Edwards ; subsidiary,"[{'head': 'PeopleSoft', 'tail': 'JD Edwards', 'relation': 'subsidiary'}]","[{'head': 'SAP AG', 'tail': 'NYSE', 'relation': 'stock_exchange'}, {'head': 'Oracle Corporation', 'tail': 'NYSE', 'r...",False,
1,1,finred1,"The Daily Show with Trevor Noah premieres tonight... and while the show will be based on Comedy Central, Viacom pans...",VH1 ; Viacom ; owned_by,"[{'head': 'VH1', 'tail': 'Viacom', 'relation': 'owned_by'}]","[{'head': 'Comedy Central', 'tail': 'Viacom', 'relation': 'owned_by'}, {'head': 'VH1', 'tail': 'Viacom', 'relation':...",False,
2,2,finred2,"""Our results for the quarter show very balanced performance across our business lines,"" said Citigroup chief executi...",Michael Corbat ; Citigroup ; employer,"[{'head': 'Michael Corbat', 'tail': 'Citigroup', 'relation': 'employer'}]","[{'head': 'Michael Corbat', 'tail': 'Citigroup', 'relation': 'chief_executive_officer'}]",False,
3,3,finred3,"Saudi Arabian budget carrier flynas, which made its first profit this year, is in talks with plane manufacturers Air...",Airbus ; aircraft ; product_or_material_produced,"[{'head': 'Airbus', 'tail': 'aircraft', 'relation': 'product_or_material_produced'}]","[{'head': 'flynas', 'tail': 'Saudi Arabian budget carrier', 'relation': 'legal_form'}, {'head': 'flynas', 'tail': 'A...",False,
4,4,finred4,"First Eagle is currently owned by members of the Arnhold family, private equity firm TA Associates Management and em...",TA Associates ; private equity ; industry,"[{'head': 'TA Associates', 'tail': 'private equity', 'relation': 'industry'}]","[{'head': 'First Eagle', 'tail': 'members of the Arnhold family', 'relation': 'owned_by'}, {'head': 'First Eagle', '...",False,
